# On-sky utilization: interactive Visit Timeline

Interactive (Bokeh) version of the "Visit Timeline" figure from
[lsst/sp_notebooks](https://github.com/lsst/sp_notebooks) `on-sky_utilization.ipynb`
(the figure shown as "Efficiency Evolution: Getting Bluer!" at the Joint Operations Review 2026).
The exact reproduction of the original lives in `on_sky_utilization_repro.ipynb`; the shared
logic (ConsDB query, categorization, twilight, colors) lives in `../utils.py` and the downtime
annotation layer in `../downtime.py` (data: `../downtime_annotations.csv`).

Each night (`day_obs`) is a column; each visit is a mark at its time of night. Grey bands are the
-12 to -18 deg solar elevation twilight (light) and brighter than -12 deg (dark).

What this version adds:

- zoom, pan, box zoom, reset; hover per visit; legend entries toggle categories;
- `EXCLUSIVE_CATEGORIES`: one category per visit, priority Focus and Alignment > Survey
  Programs > Engineering (the original draws BLOCK-T539 visits twice, orange then green);
- `DRAW_BARS`: draw each visit from exposure start to end instead of a tick at the midpoint;
- `X_AXIS`: `"continuous"` (one column per calendar night, closed nights are visible gaps, this
  matches the original's numeric axis) or `"categorical"` (only nights with data, compact);
- `Y_REF`: `"utc"` (hours after 00:00 UTC on the morning side of the night, the original's
  reference; 0 is 21:00 CLST / 20:00 CLT, i.e. early evening) or `"local"` (Chile clock time,
  each night shifted by its own UTC offset, so there is a 1 h step at the daylight-saving change);
- `ANNOTATE_DOWNTIME`: overlay the downtime spans of `../downtime_annotations.csv` (pale bands,
  a colour strip and staggered labels above the data, hover with the FRACAS tickets and notes,
  a second legend, the "Start of the LSST" marker). `DOWNTIME_MIN_NIGHTS`,
  `DOWNTIME_CATEGORIES`, `DOWNTIME_LABEL_CATEGORIES` and `DOWNTIME_LABEL_FLAG` select what is
  drawn and what is labelled (the CSV column `talk` flags the five spans curated for the RASAB
  2026 slide); `UNKNOWN` rows are never drawn unless asked for;
- `SURVEY_PROGRAMS` and `FOCUS_PROGRAMS` are parameters, with a check of what else was
  observed in the range.

All switches default to the original behavior except `ANNOTATE_DOWNTIME`, so the two can be
compared. See `../README.md` for definitions and caveats.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Parameters
day_obs_min = 20251026  # start of Early Operations
day_obs_max = 20260713  # last night before the July 2026 storm (20260714 has 3 visits only)

# Category rules. Defaults are the original notebook's hard-coded lists.
SURVEY_PROGRAMS = ["BLOCK-407", "BLOCK-408", "BLOCK-416", "BLOCK-417", "BLOCK-419", "BLOCK-421"]
FOCUS_PROGRAMS = ["BLOCK-T539"]

# Improvements, each behind a switch. False / "continuous" / "utc" = original behavior.
EXCLUSIVE_CATEGORIES = False  # True: Focus > Survey > Engineering, one layer per visit
DRAW_BARS = False  # True: exposure start to end instead of a midpoint tick
X_AXIS = "continuous"  # "continuous" (calendar nights, gaps visible) or "categorical"
Y_REF = "utc"  # "utc": hours after 00:00 UTC (original); "local": Chile clock time (CLT / CLST)
Y_RANGE = None  # (low, high) in hours in the Y_REF frame, or None for twilight-based automatic limits

# Downtime annotation layer (../downtime_annotations.csv, drawn by ../downtime.py).
ANNOTATE_DOWNTIME = True
DOWNTIME_MIN_NIGHTS = 1  # talk slide: 2
DOWNTIME_CATEGORIES = None  # None = every category but "unknown"; talk slide: ["fault", "weather", "mixed"]
DOWNTIME_LABEL_CATEGORIES = None  # None = label every drawn span; talk slide: ["fault"]
DOWNTIME_LABEL_FLAG = None  # None, or a 0/1 CSV column that gates the labels; talk slide: "talk"

CLIENT_PREFER = "token"  # "token" on the RSP / Nublado, "postgres" on an SDF login node
USE_CACHE = True
INSTRUMENT = "lsstcam"

In [ ]:
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
import downtime
import utils

logging.getLogger("numexpr").setLevel(logging.WARNING)
logging.getLogger("utils").setLevel(logging.INFO)

PLOTS_DIR = Path("../plots")
CACHE_DIR = Path("../cache")
DOWNTIME_CSV = Path("../downtime_annotations.csv")
PLOTS_DIR.mkdir(exist_ok=True)

## Data

In [ ]:
client = utils.get_consdb_client(prefer=CLIENT_PREFER)
visits_raw = utils.fetch_visits(
    client, day_obs_min, day_obs_max, instrument=INSTRUMENT, cache_dir=CACHE_DIR, use_cache=USE_CACHE
)
print(len(visits_raw), "visits on", visits_raw["day_obs"].nunique(), "nights")

In [ ]:
visits = utils.categorize_visits(
    visits_raw, SURVEY_PROGRAMS, FOCUS_PROGRAMS, exclusive=EXCLUSIVE_CATEGORIES
)
visits = utils.add_time_columns(visits)
twilight = utils.twilight_table(day_obs_min, day_obs_max)
colors = utils.rubin_category_colors()
print("layer sizes:", {k: int(visits[k].sum()) for k in ("is_engineering", "is_survey", "is_focus")})
print("colors:", colors)

### Which visits change category when the rules are made exclusive?

With the original rules a visit can be in two layers. The table lists what moves when
`EXCLUSIVE_CATEGORIES=True`, independent of the current switch setting.

In [ ]:
utils.category_changes(visits_raw, SURVEY_PROGRAMS, FOCUS_PROGRAMS)

### Is the survey program list complete?

Programs observed in the range that are **not** in `SURVEY_PROGRAMS`. `looks_like_survey` flags
`BLOCK-<digits>` names (scheduler survey blocks are named without a `T`), so any flagged row is a
candidate for the list. The second table restricts to the LSST period that started on 20260629.

In [ ]:
programs = utils.programs_in_range(visits_raw, survey_programs=SURVEY_PROGRAMS, focus_programs=FOCUS_PROGRAMS)
programs[~programs["in_survey_list"] & programs["looks_like_survey"]]

In [ ]:
utils.programs_in_range(visits_raw, 20260629, survey_programs=SURVEY_PROGRAMS, focus_programs=FOCUS_PROGRAMS)

## Interactive figure

Everything Bokeh is built inside `build_timeline()` so the figure can be saved to HTML and shown
inline from two independent documents (a Bokeh model may belong to only one document).

With `ANNOTATE_DOWNTIME` the data area ends at the top of the automatic (or given) y range, a thin
line marks that edge, and the downtime strips and labels are drawn above it (the y range is
extended to make room). Hover over a band or strip to see the CSV row. The downtime legend is
below the plot; the visit legend stays inside.

In [ ]:
from bokeh.io import output_file, output_notebook, reset_output, save, show
from bokeh.models import (
    ColumnDataSource,
    DatetimeTickFormatter,
    FixedTicker,
    HoverTool,
    Legend,
    LegendItem,
    Range1d,
)
from bokeh.plotting import figure

DAY_MS = 86_400_000.0
TWILIGHT_COLS = ("evening_nautical_h", "evening_astronomical_h", "morning_astronomical_h", "morning_nautical_h")
Y_LABEL = {
    "utc": "Time (hours after 00:00 UTC, morning side of the night)",
    "local": "Local clock time (Chile, CLT / CLST)",
}


def to_frame(visits, twilight, y_ref):
    """Copies of visits and twilight with the time-of-night columns in the chosen frame."""
    if y_ref == "utc":
        return visits, twilight
    if y_ref != "local":
        raise ValueError(f"Y_REF must be 'utc' or 'local', got {y_ref!r}")
    v, tw = visits.copy(), twilight.copy()
    for c in ("t_mid_h", "t_start_h", "t_end_h"):
        v[c] = utils.to_local_hours(v[c], v["day_obs"])
    for c in TWILIGHT_COLS:
        tw[c] = utils.to_local_hours(tw[c], tw["day_obs"])
    return v, tw


def local_ticks(y_range):
    """Even-hour tick positions and HH:00 labels for a local-time y axis."""
    ticks = np.arange(np.ceil(y_range[0] / 2) * 2, y_range[1] + 0.01, 2)
    return ticks, {float(t): f"{int(t) % 24:02d}:00" for t in ticks}


def _x_coordinates(visits, twilight, x_axis):
    """Return (x for visits, x for twilight rows, column width, axis setup dict)."""
    if x_axis == "continuous":
        xv = visits["night_utc"].astype("int64").to_numpy() / 1e6  # ms since epoch
        xt = twilight["night_utc"].astype("int64").to_numpy() / 1e6
        return xv, xt, DAY_MS, {"x_axis_type": "datetime"}
    if x_axis == "categorical":
        nights = np.sort(visits["day_obs"].unique())
        pos = {d: i for i, d in enumerate(nights)}
        xv = visits["day_obs"].map(pos).to_numpy(dtype=float)
        keep = twilight["day_obs"].isin(nights)
        xt = twilight.loc[keep, "day_obs"].map(pos).to_numpy(dtype=float)
        return xv, xt, 1.0, {"x_axis_type": "linear", "_nights": nights, "_keep": keep.to_numpy()}
    raise ValueError(f"X_AXIS must be 'continuous' or 'categorical', got {x_axis!r}")


def downtime_kwargs(**overrides):
    kw = dict(
        min_duration_days=DOWNTIME_MIN_NIGHTS,
        categories=DOWNTIME_CATEGORIES,
        label_categories=DOWNTIME_LABEL_CATEGORIES,
        label_flag=DOWNTIME_LABEL_FLAG,
    )
    kw.update(overrides)
    return kw


def build_timeline(
    visits,
    twilight,
    colors,
    exclusive=EXCLUSIVE_CATEGORIES,
    draw_bars=DRAW_BARS,
    x_axis=X_AXIS,
    y_ref=Y_REF,
    y_range=Y_RANGE,
    annotate=ANNOTATE_DOWNTIME,
    width=1500,
    height=650,
    title=None,
    tick_size=6,
    **downtime_overrides,
):
    visits, twilight = to_frame(visits, twilight, y_ref)
    xv, xt, unit, axis = _x_coordinates(visits, twilight, x_axis)
    tw = twilight if x_axis == "continuous" else twilight.loc[axis["_keep"]]

    if y_range is None:
        y_range = (
            float(np.floor((tw["evening_nautical_h"].min() - 0.5) * 2) / 2),
            float(np.ceil((tw["morning_nautical_h"].max() + 0.5) * 2) / 2),
        )

    p = figure(
        width=width,
        height=height,
        x_axis_type=axis["x_axis_type"],
        y_range=Range1d(*y_range),
        tools="xpan,xwheel_zoom,box_zoom,wheel_zoom,reset,save",
        active_scroll="xwheel_zoom",
        title=title,
        output_backend="webgl",
    )
    p.toolbar.logo = None
    p.yaxis.axis_label = Y_LABEL[y_ref]
    p.xaxis.axis_label = "Night (day_obs)"
    p.xgrid.visible = False
    p.ygrid.grid_line_alpha = 0.3
    p.outline_line_color = None
    if y_ref == "local":
        ticks, labels = local_ticks(y_range)
        p.yaxis.ticker = FixedTicker(ticks=list(ticks))
        p.yaxis.major_label_overrides = labels

    if x_axis == "continuous":
        p.xaxis.formatter = DatetimeTickFormatter(days="%Y%m%d", months="%Y%m%d", years="%Y%m%d")
    else:
        nights = axis["_nights"]
        p.xaxis.ticker = FixedTicker(ticks=list(range(len(nights))))
        p.xaxis.major_label_overrides = {i: str(d) for i, d in enumerate(nights)}
        p.xaxis.major_label_orientation = np.pi / 2
        p.xaxis.major_label_text_font_size = "7px"

    # Twilight bands. Dark = brighter than -12 deg, light = -12 to -18 deg. Drawn per night.
    # The dark band stops exactly at the top of the data area so the annotation area stays white.
    tsrc = ColumnDataSource(
        dict(
            x=xt,
            ev_naut=tw["evening_nautical_h"].to_numpy(),
            ev_astro=tw["evening_astronomical_h"].to_numpy(),
            mo_astro=tw["morning_astronomical_h"].to_numpy(),
            mo_naut=tw["morning_nautical_h"].to_numpy(),
            lo=np.full(len(tw), y_range[0] - 1),
            hi=np.full(len(tw), y_range[1]),
        )
    )
    band_kw = dict(x="x", width=unit, source=tsrc, line_color=None, level="underlay")
    p.vbar(bottom="lo", top="ev_naut", fill_color="black", fill_alpha=0.4, **band_kw)
    p.vbar(bottom="mo_naut", top="hi", fill_color="black", fill_alpha=0.4, **band_kw)
    tw_light = p.vbar(bottom="ev_naut", top="ev_astro", fill_color="black", fill_alpha=0.2, **band_kw)
    p.vbar(bottom="mo_astro", top="mo_naut", fill_color="black", fill_alpha=0.2, **band_kw)

    # Visits, one renderer per layer, drawn in the original order (last on top).
    layers = [
        ("Engineering", "is_engineering"),
        ("Survey Programs", "is_survey"),
        ("Focus and Alignment", "is_focus"),
    ]
    hover_cols = [
        "visit_id", "day_obs_str", "local_mid", "utc_mid", "science_program",
        "observation_reason", "img_type", "band", "exp_time", "psf_fwhm_arcsec", "category",
    ]
    legend_items = []
    renderers = []
    for name, flag in layers:
        sel = visits[flag].to_numpy()
        if not sel.any():
            continue
        sub = visits.loc[sel]
        data = {c: sub[c].to_numpy() for c in hover_cols}
        data["x"] = xv[sel]
        data["y"] = sub["t_mid_h"].to_numpy()
        data["y0"] = sub["t_start_h"].to_numpy()
        data["y1"] = sub["t_end_h"].to_numpy()
        src = ColumnDataSource(data)
        if draw_bars:
            r = p.vbar(
                x="x", width=0.8 * unit, bottom="y0", top="y1", source=src,
                fill_color=colors[name], line_color=colors[name], line_width=0.5,
            )
        else:
            # Fixed screen-size tick, like the original matplotlib "_" marker (WebGL accelerated).
            r = p.scatter(x="x", y="y", source=src, marker="dash", size=tick_size, line_color=colors[name], line_width=1.5)
        renderers.append(r)
        legend_items.append(LegendItem(label=f"{name} ({sel.sum():,})", renderers=[r]))

    legend_items.append(LegendItem(label="-12 to -18 deg twilight", renderers=[tw_light]))
    legend = Legend(items=legend_items, location="top_left", click_policy="hide", label_text_font_size="10px")
    p.add_layout(legend)

    hover = HoverTool(
        renderers=renderers,
        tooltips=[
            ("visit", "@visit_id"),
            ("day_obs", "@day_obs_str"),
            ("local", "@local_mid"),
            ("UTC", "@utc_mid"),
            ("program", "@science_program"),
            ("reason", "@observation_reason"),
            ("img_type / band", "@img_type / @band"),
            ("exp_time", "@exp_time{0.0} s"),
            ("PSF FWHM", "@psf_fwhm_arcsec{0.00} arcsec"),
            ("category", "@category"),
        ],
        point_policy="snap_to_data",
    )
    p.add_tools(hover)

    if annotate:
        if x_axis != "continuous":
            print("downtime annotations need X_AXIS='continuous'; skipped")
        else:
            downtime.annotate_downtime_bokeh(
                p,
                DOWNTIME_CSV,
                day_obs_min=day_obs_min,
                day_obs_max=day_obs_max,
                y_top=y_range[1],
                frame_width_px=width - 110,
                **downtime_kwargs(**downtime_overrides),
            )
    return p


def render(build, stem, title):
    """Save to ../plots/<stem>.html and show inline, from two independent documents."""
    reset_output()
    output_file(PLOTS_DIR / f"{stem}.html", title=title)
    path = save(build())
    reset_output()
    output_notebook()
    show(build())
    return path

In [ ]:
variant = "-".join(
    [
        "exclusive" if EXCLUSIVE_CATEGORIES else "original",
        "bars" if DRAW_BARS else "ticks",
        X_AXIS,
        Y_REF,
    ]
    + (["annotated"] if ANNOTATE_DOWNTIME else [])
)
stem = f"on_sky_utilization_interactive_{variant}_{day_obs_min}-{day_obs_max}"
title = f"On-sky utilization {day_obs_min} to {day_obs_max} ({variant})"
html_path = render(lambda: build_timeline(visits, twilight, colors, title=title), stem, title)
print("saved", html_path)

## Static snapshots

Matplotlib renders of the same data and switches, for slides and for the README, at 16:9 slide
size with 11 pt fonts. Two are made: the current switch settings, and the fully improved variant
(exclusive categories, bars, continuous axis) for a side-by-side with the reproduction. The
downtime layer uses the same `downtime.annotate_downtime` as `../make_utilization_annotated.py`,
which is the script that produces the talk figures (`plots/utilization_annotated*.png`).

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from lsst.utils.plotting import publication_plots

publication_plots.set_rubin_plotstyle()

SLIDE_FIGSIZE = (13.3, 7.5)
SLIDE_FONT = 11


def static_snapshot(
    visits_raw, twilight, colors, exclusive, draw_bars, x_axis, y_range, path, title="",
    y_ref=Y_REF, annotate=ANNOTATE_DOWNTIME, **downtime_overrides,
):
    v = utils.add_time_columns(
        utils.categorize_visits(visits_raw, SURVEY_PROGRAMS, FOCUS_PROGRAMS, exclusive=exclusive)
    )
    v, tw = to_frame(v, twilight, y_ref)
    xv, xt, unit, axis = _x_coordinates(v, tw, x_axis)
    tw = tw if x_axis == "continuous" else tw.loc[axis["_keep"]]
    if x_axis == "continuous":  # matplotlib date numbers are days since 1970-01-01
        xv, xt, unit = xv / DAY_MS, xt / DAY_MS, 1.0
    if y_range is None:
        y_range = (tw["evening_nautical_h"].min() - 0.5, tw["morning_nautical_h"].max() + 0.5)

    plt.rcParams.update({"font.size": SLIDE_FONT})
    fig, ax = plt.subplots(figsize=SLIDE_FIGSIZE)
    lo, hi = y_range[0] - 1, y_range[1] + 1
    for b, t, a in [
        (lo, tw["evening_nautical_h"], 0.4),
        (tw["evening_nautical_h"], tw["evening_astronomical_h"], 0.2),
        (tw["morning_astronomical_h"], tw["morning_nautical_h"], 0.2),
        (tw["morning_nautical_h"], hi, 0.4),
    ]:
        ax.bar(xt, np.asarray(t) - np.asarray(b), bottom=b, width=unit, color="black", alpha=a, linewidth=0, zorder=1)
    for name, flag in [("Engineering", "is_engineering"), ("Survey Programs", "is_survey"), ("Focus and Alignment", "is_focus")]:
        sel = v[flag].to_numpy()
        sub = v.loc[sel]
        if draw_bars:
            ax.bar(xv[sel], (sub["t_end_h"] - sub["t_start_h"]).to_numpy(), bottom=sub["t_start_h"].to_numpy(),
                   width=0.8 * unit, color=colors[name], linewidth=0.3, edgecolor=colors[name],
                   label=f"{name} ({sel.sum():,})", zorder=2)
        else:
            ax.hlines(sub["t_mid_h"].to_numpy(), xv[sel] - 0.4 * unit, xv[sel] + 0.4 * unit, colors=colors[name],
                      linewidth=0.9, label=f"{name} ({sel.sum():,})", zorder=2)
    ax.set_ylim(*y_range)
    ax.set_xlim(xt.min() - unit, xt.max() + unit)
    ax.minorticks_off()
    if x_axis == "continuous":
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
        ax.xaxis.set_minor_locator(mdates.DayLocator(bymonthday=[8, 15, 22]))
        ax.tick_params(axis="x", which="minor", length=3)
    else:
        nights = axis["_nights"]
        ax.set_xticks(np.arange(len(nights)), [str(d) for d in nights], rotation=90, fontsize=6)
    if y_ref == "local":
        ticks, labels = local_ticks(y_range)
        ax.set_yticks(ticks, [labels[float(t)] for t in ticks])
    ax.set_ylabel(Y_LABEL[y_ref].replace(", morning side of the night", ""), fontsize=SLIDE_FONT + 1)
    ax.set_xlabel("Night (day_obs)", fontsize=SLIDE_FONT + 1)
    ax.legend(loc="upper left", fontsize=SLIDE_FONT, framealpha=0.95)
    if annotate and x_axis == "continuous":
        # Title goes in the figure corner: the space above the axes belongs to the labels.
        fig.text(0.01, 0.99, title, ha="left", va="top", fontsize=SLIDE_FONT, color="#6A6E6E")
        fig.subplots_adjust(left=0.065, right=0.99, bottom=0.2, top=0.93)
        downtime.annotate_downtime(ax, DOWNTIME_CSV, label_fontsize=SLIDE_FONT, **downtime_kwargs(**downtime_overrides))
    else:
        if annotate:
            print("downtime annotations need X_AXIS='continuous'; skipped")
        ax.set_title(title, fontsize=SLIDE_FONT)
        fig.tight_layout()
    fig.savefig(path, dpi=300)
    plt.show()
    return path


static_snapshot(
    visits_raw, twilight, colors, EXCLUSIVE_CATEGORIES, DRAW_BARS, X_AXIS, Y_RANGE,
    PLOTS_DIR / f"on_sky_utilization_{variant}_{day_obs_min}-{day_obs_max}.png",
    title=f"On-sky utilization {day_obs_min} to {day_obs_max} ({variant})",
)
static_snapshot(
    visits_raw, twilight, colors, True, True, "continuous", Y_RANGE,
    PLOTS_DIR / f"on_sky_utilization_exclusive-bars-continuous-{Y_REF}_{day_obs_min}-{day_obs_max}.png",
    title=f"On-sky utilization {day_obs_min} to {day_obs_max} (exclusive categories, exposure bars, continuous axis)",
);

## Per-night summary table

Hours of visits per category and night, from exposure durations, so the figure can be read
without hovering. Saved next to the plots as CSV.

In [ ]:
excl = utils.add_time_columns(utils.categorize_visits(visits_raw, SURVEY_PROGRAMS, FOCUS_PROGRAMS, exclusive=True))
excl["exp_hours"] = (excl["t_end_h"] - excl["t_start_h"])
nightly = (
    excl.pivot_table(index="day_obs", columns="category", values="exp_hours", aggfunc="sum", observed=False)
    .fillna(0.0)
    .round(2)
)
nightly["n_visits"] = excl.groupby("day_obs").size()
nightly = nightly.join(twilight.set_index("day_obs")[["evening_nautical_h", "morning_nautical_h"]], how="left")
nightly["night_length_h"] = (nightly["morning_nautical_h"] - nightly["evening_nautical_h"]).round(2)
nightly.to_csv(PLOTS_DIR / f"nightly_summary_{day_obs_min}-{day_obs_max}.csv")
nightly.tail(20)